# 02 - Total p-Variation (TpV) Reconstruction for Sparse-View CT

This notebook implements a model-based variational reconstruction method for sparse-view CT. The measured data are the degraded Mayo2 sinograms generated in notebook 00, and the reconstruction is obtained by solving the unconstrained optimization problem:

$$
\hat{x} = \arg\min_x \frac{1}{2}\|Kx-y^\delta\|_2^2 + \lambda R_{TpV}(x)
$$

where:
- $K$ is the parallel-beam CT projector (Radon transform).
- $y^\delta$ is the noisy sinogram.
- $R_{TpV}(x) = \sum_i \|\nabla x_i\|_2^p$ is the Total $p$-Variation prior, with $p$ in the non-convex sparse range $0.1 < p < 0.5$ as requested by the project specifications.

The solver is based on the **Chambolle-Pock Primal-Dual algorithm** (using `IPPy` tools) initialized from scratch (zeros) without any FBP initialization, to focus strictly on the variational paradigm. 

To comply with the project specifications (*"Students must ensure all methods use the same degraded inputs"*), **both this notebook and the ResUNet notebook evaluate on the exact same central test slice**, ensuring a perfectly fair and mathematically rigorous comparison between the variational and deep learning models.

## 1. Environment Setup and Imports

Mount the Google Drive, install `astra-toolbox` if needed, and import libraries from standard Python, PyTorch, and `IPPy`.

In [ ]:
!pip install astra-toolbox

from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

# Paths setup
PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
PROCESSED_DIR = PROJECT_ROOT / "processed2"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tpv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, solvers, utilities
from IPPy.utilities import normalize

# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device configuration (CPU or GPU)
device = utilities.get_device()
print("Device used:", device)
print("Processed data directory:", PROCESSED_DIR)
print("TpV outputs directory:", OUTPUT_DIR)

## 2. Load the Preprocessed Test Data Contract

Load the first test patient file reserved for final multi-view TpV reporting.

In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

def load_first_processed_patient(split_name: str) -> dict:
    split_info = manifest["splits"][split_name]
    patient_record = split_info["patients"][0]
    patient_path = PROCESSED_DIR / patient_record["path"]
    payload = torch.load(patient_path, map_location="cpu")
    return {
        "split": split_name,
        "path": patient_path,
        "clean": payload["clean"].to(torch.float32),
        "sinograms": {key: val.to(torch.float32) for key, val in payload["sinograms"].items()},
        "metadata": payload["metadata"],
    }

test_data = load_first_processed_patient("test")

metadata = test_data["metadata"]
print(f"Loaded {test_data['split']} patient ID: {metadata['patient_id']}")
print(f"  clean images shape: {tuple(test_data['clean'].shape)}")
for views_key, sino in test_data["sinograms"].items():
    print(f"  sinogram ({views_key} views) shape: {tuple(sino.shape)}")

## 3. Configure CT Projector and TpV Solver

Define the number of views to test (select from 180, 90, 60, 45) and initialize the parallel-beam CT projector $K$ and the unconstrained Chambolle-Pock TpV solver.

In [ ]:
# Sparse-view settings to test
ANGLE_COUNTS = (180, 90, 60, 45)
IMAGE_SIZE = 256
DETECTOR_SIZE = 256
GEOMETRY = "parallel"
angle_keys = [str(n_views) for n_views in ANGLE_COUNTS]

available_test_angles = set(test_data["sinograms"].keys())
missing_angles = [angle_key for angle_key in angle_keys if angle_key not in available_test_angles]
if missing_angles:
    available = ", ".join(sorted(available_test_angles, key=int))
    missing = ", ".join(missing_angles)
    raise ValueError(f"Missing test angle config(s): {missing}. Available configs: {available}")

solver_device = torch.device("cpu")

# Initialize one physical parallel-beam projector and TpV solver for each view setting.
projectors = {
    angle_key: operators.CTProjector(
        img_shape=(IMAGE_SIZE, IMAGE_SIZE),
        angles=np.linspace(0.0, np.pi, int(angle_key)),
        det_size=DETECTOR_SIZE,
        geometry=GEOMETRY,
        force_cpu=True,  # Set to False if CUDA/GPU support is fully configured in ASTRA
    )
    for angle_key in angle_keys
}

tpv_solvers = {
    angle_key: solvers.ChambollePockTpVUnconstrained(projector)
    for angle_key, projector in projectors.items()
}

print("Initialized CT projectors and TpV solvers for views:", ", ".join(angle_keys))
print("TpV solver device:", solver_device)

## 4. Multi-View Final Test Evaluation

Run this cell to select the shared test slice, apply the Chambolle-Pock solver for every sparse-view configuration, and read PSNR/SSIM from the solver diagnostics.

In [ ]:
# -------------------- FINAL TEST HYPERPARAMETERS --------------------
lmbda = 0.01          # Regularization parameter (weight of TpV penalty)
p = 0.35              # Sparsity parameter (strictly between 0.1 and 0.5)
maxiter = 150         # Number of maximum iterations
# -------------------------------------------------------------------------------

if not (0.1 < p < 0.5):
    raise ValueError(f"Project specifications require 0.1 < p < 0.5. Got p = {p}")

def move_info_to_cpu(info: dict) -> dict:
    return {
        key: value.detach().cpu() if isinstance(value, torch.Tensor) else value
        for key, value in info.items()
    }

def run_tpv_reconstruction(split_data: dict, angle_key: str, solver) -> dict:
    clean_images = split_data["clean"]
    sinograms = split_data["sinograms"]
    split_name = split_data["split"]

    # Select the fixed central slice for consistency across all view settings.
    slice_idx = clean_images.shape[0] // 2
    print(f"\nRunning {split_name} reconstruction for {angle_key} views on central slice index: {slice_idx}")

    x_true = clean_images[slice_idx : slice_idx + 1].to(solver_device)
    y_delta = sinograms[angle_key][slice_idx : slice_idx + 1].to(solver_device)

    x_sol, info = solver(
        y_delta,
        lmbda=lmbda,
        starting_point=None,  # Standard zero-image initialization
        x_true=x_true,
        maxiter=maxiter,
        tolf=1e-5,
        tolx=1e-5,
        p=p,
        verbose=True,
    )

    info = move_info_to_cpu(info)
    x_sol = x_sol.detach().cpu()
    x_true_cpu = x_true.detach().cpu()
    x_sol_norm = normalize(x_sol).clamp(0.0, 1.0)

    psnr_val = float(info["PSNR"][-1, 0])
    ssim_val = float(info["SSIM"][-1, 0])

    print(f"{split_name} PSNR: {psnr_val:.4f} dB")
    print(f"{split_name} SSIM: {ssim_val:.4f}")

    return {
        "split": split_name,
        "angle_key": angle_key,
        "slice_idx": slice_idx,
        "x_true_cpu": x_true_cpu,
        "x_sol_norm": x_sol_norm,
        "info": info,
        "iterations": info["iterations"],
        "psnr": psnr_val,
        "ssim": ssim_val,
    }

tpv_results = {
    angle_key: run_tpv_reconstruction(test_data, angle_key, tpv_solvers[angle_key])
    for angle_key in angle_keys
}

# Keep a final active result for compatibility with downstream visualization/reporting cells.
test_result = tpv_results[str(ANGLE_COUNTS[-1])]
active_result = test_result
active_split = active_result["split"]
angle_key = active_result["angle_key"]
n_views = int(angle_key)
slice_idx = active_result["slice_idx"]
x_true_cpu = active_result["x_true_cpu"]
x_sol_norm = active_result["x_sol_norm"]
psnr_val = active_result["psnr"]
ssim_val = active_result["ssim"]

## 5. Visual Inspection and Error Map

Plot the Ground Truth image, the TpV reconstruction, and the absolute error map $|TpV - GT|$ to inspect the spatial distribution of reconstruction errors. The generated visual comparison is saved in the `/outputs/tpv/` directory.

In [ ]:
def info_series(info: dict, key: str) -> np.ndarray:
    values = info[key].detach().cpu().numpy() if isinstance(info[key], torch.Tensor) else np.asarray(info[key])
    return np.atleast_1d(np.squeeze(values)).astype(float)

def positive_for_log(values: np.ndarray) -> np.ndarray:
    return np.clip(values, 1e-12, None)

for angle_key in angle_keys:
    result = tpv_results[angle_key]
    current_n_views = int(angle_key)
    current_slice_idx = result["slice_idx"]
    current_info = result["info"]

    gt_np = result["x_true_cpu"][0, 0].numpy()
    recon_np = result["x_sol_norm"][0, 0].numpy()
    error_map = np.abs(recon_np - gt_np)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
    fig.suptitle(
        f"TpV Test Reconstruction Panel - {current_n_views} views - Slice {current_slice_idx}\n"
        f"(lmbda={lmbda}, p={p}, iterations={result['iterations']})",
        fontsize=14,
    )

    axes[0].imshow(gt_np, cmap="gray", vmin=0.0, vmax=1.0)
    axes[0].set_title("Ground Truth")
    axes[0].axis("off")

    axes[1].imshow(recon_np, cmap="gray", vmin=0.0, vmax=1.0)
    axes[1].set_title(f"TpV Reconstruction\nPSNR: {result['psnr']:.2f} dB | SSIM: {result['ssim']:.4f}")
    axes[1].axis("off")

    im_err = axes[2].imshow(error_map, cmap="magma")
    axes[2].set_title("Absolute Error Map |TpV - GT|")
    axes[2].axis("off")
    fig.colorbar(im_err, ax=axes[2], shrink=0.8)

    output_fig_path = OUTPUT_DIR / f"tpv_test_reconstruction_panel_{angle_key}.png"
    fig.savefig(output_fig_path, dpi=150)
    plt.show()
    plt.close(fig)
    print("Saved TpV reconstruction panel:", output_fig_path)

    iteration_axis = np.arange(1, len(info_series(current_info, "obj")) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    fig.suptitle(f"TpV Convergence - {current_n_views} views - Slice {current_slice_idx}", fontsize=14)

    axes[0, 0].plot(iteration_axis, positive_for_log(info_series(current_info, "obj")))
    axes[0, 0].set_title("Objective")
    axes[0, 0].set_yscale("log")
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(iteration_axis, positive_for_log(info_series(current_info, "residues")))
    axes[0, 1].set_title("Residual")
    axes[0, 1].set_yscale("log")
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(iteration_axis, info_series(current_info, "PSNR"))
    axes[1, 0].set_title("PSNR")
    axes[1, 0].set_xlabel("Iteration")
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(iteration_axis, info_series(current_info, "SSIM"))
    axes[1, 1].set_title("SSIM")
    axes[1, 1].set_xlabel("Iteration")
    axes[1, 1].grid(True, alpha=0.3)

    for ax in axes.flat:
        if len(iteration_axis) == 1:
            ax.set_xlim(0.5, 1.5)
        else:
            ax.set_xlim(1, len(iteration_axis))

    convergence_fig_path = OUTPUT_DIR / f"tpv_convergence_{angle_key}.png"
    fig.savefig(convergence_fig_path, dpi=150)
    plt.show()
    plt.close(fig)
    print("Saved TpV convergence panel:", convergence_fig_path)

## 6. Quantitative Results

Display the required PSNR and SSIM metrics for the TpV reconstruction.

In [ ]:
print("="*72)
print(f"{ 'VIEWS':<8} | { 'METHOD':<12} | { 'ITERATIONS':<10} | { 'PSNR (dB)':<12} | { 'SSIM':<8}")
print("="*72)
for angle_key in angle_keys:
    result = tpv_results[angle_key]
    print(
        f"{ angle_key:<8} | { 'TpV test':<12} | { result['iterations']:<10} | "
        f"{ result['psnr']:<12.4f} | { result['ssim']:<8.4f}"
    )
print("="*72)